<a href="https://colab.research.google.com/github/bomgom02-netizen/skills_claude/blob/main/Get_trading_signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

In [ ]:
import yfinance as yf
import requests
import time

# ==========================================
# 🛡️ [설정] 텔레그램 및 감시 대상
# ==========================================
TOKEN = '7796181604:AAGo0xhEHPIR6aza8vsMVAVEINdI1_4f08k'
CHAT_ID = '8528061505'

# 감시할 종목 상세 정보 (티커, 매수/매도 기준가)
targets = {
    'ARM': {'ticker': 'ARM', 'sell_price': 155.0},
    'BITU': {'ticker': 'BITU', 'buy_price': 0}, # BITU 로직은 아래에서 별도 처리
    '우리기술': {'ticker': '032820.KS', 'sell_price': 0}, # 즉시 매도 설정
    '삼성전자': {'ticker': '005930.KS', 'buy_price': 82000},
    '한화에어로스페이스': {'ticker': '012450.KS', 'buy_price': 270000},
    '고려아연': {'ticker': '010130.KS', 'buy_price': 500000}
}

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
    params = {'chat_id': CHAT_ID, 'text': message}
    try:
        requests.get(url, params=params)
    except Exception as e:
        print(f"❌ 텔레그램 전송 오류: {e}")

def get_trading_signals(current_prices):
    signals = []

    # 매도 시그널
    if current_prices.get('ARM', 9999) <= 155.0:
        signals.append({'ticker': 'ARM', 'action': 'SELL', 'reason': '손절가 이탈 ($155)'})

    if current_prices.get('BITU_drawdown', 0) <= -0.20:
        signals.append({'ticker': 'BITU', 'action': 'SELL', 'reason': '변동성 헤지 (-20%)'})

    signals.append({'ticker': '우리기술', 'action': 'SELL', 'reason': '자본 효율화'})

    # 매수 시그널
    if current_prices.get('삼성전자', 999999) <= 82000:
        signals.append({'ticker': '삼성전자', 'action': 'BUY', 'reason': '밸류에이션 매력'})

    if current_prices.get('한화에어로스페이스', 999999) <= 270000:
        signals.append({'ticker': '한화에어로스페이스', 'action': 'BUY', 'reason': '성장주 저점 확보'})

    if current_prices.get('고려아연', 999999) <= 500000:
        signals.append({'ticker': '고려아연', 'action': 'BUY', 'reason': '실적 모멘텀'})

    return signals

def run_monitor():
    print("🚀 실시간 통합 감시 시스템 가동 중...")
    while True:
        current_prices = {}

        # 1. 데이터 수집
        for name, info in targets.items():
            try:
                data = yf.Ticker(info['ticker']).history(period='1d')
                if not data.empty:
                    current_prices[name] = data['Close'].iloc[-1]
            except: continue

        # BITU 변동성 등 별도 데이터 추가
        current_prices['BITU_drawdown'] = -0.21 # 예시값, 실제 연동 시 로직 추가 필요

        # 2. 시그널 생성 및 알림
        signals = get_trading_signals(current_prices)
        for s in signals:
            msg = f"🔔 [{s['action']}] {s['ticker']}\n이유: {s['reason']}"
            send_telegram(msg)
            print(msg)

        time.sleep(300) # 5분 간격

if __name__ == "__main__":
    run_monitor()

🚀 실시간 통합 감시 시스템 가동 중...


ERROR:yfinance:$032820.KS: possibly delisted; no price data found  (period=1d)


🔔 [SELL] BITU
이유: 변동성 헤지 (-20%)
🔔 [SELL] 우리기술
이유: 자본 효율화


ERROR:yfinance:$032820.KS: possibly delisted; no price data found  (period=1d)


🔔 [SELL] BITU
이유: 변동성 헤지 (-20%)
🔔 [SELL] 우리기술
이유: 자본 효율화


ERROR:yfinance:$032820.KS: possibly delisted; no price data found  (period=1d)


🔔 [SELL] BITU
이유: 변동성 헤지 (-20%)
🔔 [SELL] 우리기술
이유: 자본 효율화


ERROR:yfinance:$032820.KS: possibly delisted; no price data found  (period=1d)


🔔 [SELL] BITU
이유: 변동성 헤지 (-20%)
🔔 [SELL] 우리기술
이유: 자본 효율화
